# Fase 2 — Inventario controllato delle quattro classi riservate

**Attenzione: questo notebook, a differenza dei due precedenti, legge intenzionalmente il ground truth in un componente di audit isolato.** Legge soltanto i piccoli `query.csv` e `record.csv` interni all'archivio originale `Market.zip` per verificare l'allineamento delle righe con il manifest congelato e produrre **solo conteggi aggregati** per classe. Non stampa associazioni caso–etichetta, non apre la telemetria, non lancia agenti, non esegue scoring e non scrive dati.

Eseguire **solo dopo i test PyCharm** `test_ood_class_inventory.py`. Non importare il modulo nell'agente e non aggiungere output GT ai prompt LLM.


In [ ]:
# FASE 2 — audit etichette CONTROLLATO, aggregato e ISOLATO dall'agente.
# Legge il ground truth di record.csv/scoring_points SOLO per inventario statistico.
# Nessuna chiamata Gemini/Claude, nessun output case_id -> classe, nessun file scritto.
from google.colab import drive
from pathlib import Path
import json
import os
import re
import subprocess
import sys

BRANCH = "feature/ood-knowledge-transfer-audit-2026-11-10"
REPO = Path("/content/Tesi_OpenRCA_OOD_Audit")
DATA = Path("/content/drive/MyDrive/Tesi_OpenRCA")
URL = "https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git"

drive.mount("/content/drive")
for required in (
    DATA / "Market.zip",
    DATA / "data/processed/split_market_v1.csv",
    DATA / "data/processed/benchmark_inputs_market.csv",
):
    if not required.is_file():
        raise FileNotFoundError(f"Input necessario assente: {required.name}")
if REPO.exists():
    if not (REPO / ".git").is_dir():
        raise RuntimeError("Directory del clone OOD non-Git: nessuna sovrascrittura")
    current = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True
    ).strip()
    if current != BRANCH:
        raise RuntimeError("Branch diverso: fermato senza switch/reset")
    if subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip():
        raise RuntimeError("Clone con modifiche locali: pull non eseguito")
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True, text=True, capture_output=True
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch",
         BRANCH, URL, str(REPO)],
        check=True, text=True, capture_output=True
    )

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("=== FASE 2 — INVENTARIO CLASSI OOD (GROUND TRUTH CONTROLLATO) ===")
print("Branch:", BRANCH)
print("Commit:", commit)
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO / "src")
run = subprocess.run(
    [sys.executable, "-m", "evaluation.ood_class_inventory",
     "--data-dir", str(DATA)],
    cwd=str(REPO), env=env, text=True, capture_output=True
)
if run.returncode:
    raise RuntimeError(
        "Audit controllato fallito. Nessuna inferenza eseguita.\n"
        + run.stderr[-5000:]
    )
try:
    result = json.loads(run.stdout)
except json.JSONDecodeError as exc:
    raise RuntimeError("Audit non restituisce JSON: dati NON verificati") from exc
if (
    result.get("audit") != "ood_class_inventory_controlled_gt_v1"
    or result.get("source_verified") is not True
    or result.get("alignment_checked_manifest_cases") != 148
    or result.get("ood_query_count") != 15
    or result.get("read_ground_truth") is not True
    or result.get("read_telemetry") is not False
    or result.get("files_written") is not False
    or result.get("emits_case_label_mapping") is not False
    or result.get("model_api_calls") != 0
):
    raise RuntimeError("Indicatori inattesi: inventario non validato")
dump = json.dumps(result, indent=2, ensure_ascii=False)
if re.search(r"cloudbed-[12]_q\d+", dump):
    raise RuntimeError("Rilevato case_id in output: non stampo possibili associazioni GT")
print("\n=== RISULTATI AGGREGATI PER CLASSE — NON PER CASO ===")
print(dump)
if result["classification_status"] != "complete":
    print("\nATTENZIONE: nomi delle classi non riconosciuti;")
    print("serve revisione della nomenclatura PRIMA di pianificare le esecuzioni.")
else:
    print("\nEtichette riconosciute; procedere comunque alla revisione")
    print("dell'indipendenza e del disegno discovery/test.")
print("\nZERO API Gemini/Claude. Ground truth NON disponibile agli agenti.")


## Interpretazione

I conteggi identificano query associate a record di guasto, **non** dimostrano indipendenza fra fault injection. Se esistono etichette non riconosciute, il report le mostra soltanto in forma aggregata per revisione; non usare i conteggi per assegnare discovery/test finché la nomenclatura non è verificata.
